# Towards a GIS system

In [ ]:
#| default_exp database

In [ ]:
#| export
import numpy as np
import sys
import os
import math
from math import radians, cos, sin, sqrt, atan2
import random

#data
from collections import namedtuple
from dataclasses import dataclass,  field, asdict
import json
from typing import List , Optional
from enum import Enum



In [ ]:
#| export
from dataclasses import dataclass
from typing import Optional

from datetime import datetime


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx


In [ ]:
#| export
from importlib import resources
from pathlib import Path
import tempfile
import shutil
import sqlite3
from dataclasses import dataclass
from typing import Iterator
import numpy as np

In [ ]:
#| export
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, hexBackground, HexRegion, unique_windy_edge, HexChunk
from HexMagic.styles import StyleCSS, SVGBuilder

In [ ]:
#| export
from HexMagic.geology import River, Watershed

In [ ]:
#| export
import asyncio
from dataclasses import dataclass

In [ ]:
#!cat *.md

In [ ]:
#!cat ../llms.txt

## Helpers

In [ ]:
#| export
LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

In [ ]:
#| export
@dataclass
class ChunkRef:
    """Reference to a chunk in the world - uses HexPosition for coordinates."""
    position: HexPosition  # Chunk's position in chunk-space (q+r+s=0)
    rings: int             # Hex rings per chunk
    
    @property
    def center_hex(self) -> HexPosition:
        """World hex coordinates of chunk center."""
        spacing = self.rings * 2
        return spacing * self.position
    
    @property
    def key(self) -> tuple[int, int, int]:
        """Hashable key for dict storage."""
        return (self.position.q, self.position.r, self.position.s)
    
    def __hash__(self):
        return hash(self.key)
    
    @classmethod
    def from_key(cls, key, chunk_rings: int) -> 'ChunkRef':
        if isinstance(key, tuple):
            q, r, s = key
        else:
            q, r, s = map(int, key.split(','))
        return cls(HexPosition(q, r, s), chunk_rings)




def chunk_to_world(chunk: ChunkRef, local_pos: HexPosition) -> HexPosition:
    """Map local hex position within chunk to world coordinates."""
    return chunk.center_hex + local_pos


def world_to_chunk(world_pos: HexPosition, chunk_rings: int) -> tuple[HexPosition, HexPosition]:
    """Map world position to (chunk_position, local_position_within_chunk)."""
    spacing = chunk_rings * 2
    
    chunk_q = world_pos.q // spacing
    chunk_r = world_pos.r // spacing
    chunk_s = -chunk_q - chunk_r
    
    chunk_pos = HexPosition(chunk_q, chunk_r, chunk_s)
    chunk_center = spacing * chunk_pos
    local_pos = world_pos - chunk_center
    
    return chunk_pos, local_pos


## Tables

In [ ]:
#| export
@dataclass  
class HexData:
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    grid_index: int = 0
    elevation: float = 0.0
    plate_id:int = -1
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    distance_from_coast: Optional[float] = None
    watershed_id: Optional[int] = None  # NEW: FK to WatershedMeta
    modified: int = 0


In [ ]:
#| export
@dataclass
class HexWeather:
    """Weather data for a hex at a point in time."""
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    # Core weather fields
    temperature: float = 0.0           # °C
    precipitation: float = 0.0         # mm/year
    humidity: Optional[float] = None   # %
    # Derived/computed fields
    climate_pet: Optional[float] = None          # potential evapotranspiration
    aridity_index: Optional[float] = None
    temp_seasonality: Optional[float] = None
    precip_seasonality: Optional[float] = None
    distance_from_coast: Optional[float] = None  # hex units
    # Temporal
    season: str = ""                   # e.g., "annual", "summer", "winter"

     # NEW: Chunk identification (NULL for coarse-level weather)
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    
    # NEW: For cache invalidation
    scale_level: int = 0        # 0=coarse, 1=fine chunk
    climate_name: str = ""      # e.g., "mediterranean" - invalidate if changed
    wind_dir: Optional[float] = None  # Wind direction used in computation


    modified: int = 0


In [ ]:
#| export
@dataclass
class WatershedMeta:
    """Metadata for a stored watershed."""
    id: int = None
    world_id: int = 0
    name: str = ""
    # Terminal hex (outlet)
    terminal_q: int = 0
    terminal_r: int = 0
    terminal_s: int = 0
    is_ocean: bool = True
    # Stats
    total_flow: float = 0.0
    area_hexes: int = 0
    # Encoded data
    river_tree: str = ""   # River.encode()
    style: str = ""        # StyleCSS.encode()
    created: int = 0

    # NEW: Chunk identification (NULL for coarse-level watershed)
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None

    modified: int = 0

In [ ]:
#| export


@dataclass
class TerrainWorld:
    """Metadata for a stored terrain."""
    id: int = None
    name: str = ""
    hex_radius: float = 25.0
    nrows: int = 0
    ncols: int = 0
    sea_level: float = 0.0
    elevation_delta: float = 90.0
    # Encoded extras (climate, geo bounds, etc)
    extras: str = ""
    created: int = 0
    modified: int = 0



In [ ]:
#| export
@dataclass
class User:
    username: str
    email: str
    password: str
    created: int  # Unix timestamp
    sessionID: str
    activeWorld: int # a link into world_id for world 
    id: int = None

In [ ]:
#| export
@dataclass
class ChunkBorder:
    """Track drainage across chunk boundaries."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0  
    chunk_s: int = 0
    border_hex_q: int = 0  # Local hex at border
    border_hex_r: int = 0
    border_hex_s: int = 0
    downstream_chunk_q: int = 0  # Which chunk it drains to
    downstream_chunk_r: int = 0
    downstream_chunk_s: int = 0
    flow_volume: float = 0.0  # Accumulated upstream area


## Main Storage

In [ ]:
#| export
class GeoStorage:

    def __init__(self, custom_path=None):
        path = GeoStorage.get_db_path(custom_path)
        self.path = path
        self.createDB()

    def createDB(self):
        db = database(self.path)
        self.db = db

        # Create tables using dataclasses
        db.create(TerrainWorld, pk='id', if_not_exists=True, transform=True)
        db.create(HexData, pk='id', if_not_exists=True, transform=True)
        db.create(User, pk='id', if_not_exists=True, transform=True)

        db.create(HexWeather, pk='id', if_not_exists=True, transform=True)

        # NEW: Watershed table
        db.create(WatershedMeta, pk='id', if_not_exists=True, transform=True)
        db.create(ChunkBorder, pk='id', if_not_exists=True, transform=True)

        # After creating tables
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_world ON hex_data(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_coords ON hex_data(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_grid ON hex_data(world_id, grid_index)")


        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_temporal ON hex_data(world_id, q, r, s, modified DESC)")

                # Find borders by source chunk
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_chunk ON chunk_border(world_id, chunk_q, chunk_r, chunk_s)")

        # Find borders draining INTO a chunk (for propagating flow downstream)
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_downstream ON chunk_border(world_id, downstream_chunk_q, downstream_chunk_r, downstream_chunk_s)")


        

         # NEW: Index for watershed queries
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_watershed ON hex_data(world_id, watershed_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_watershed_world ON watershed_meta(world_id)")

        # Indices for weather queries
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_world ON hex_weather(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_coords ON hex_weather(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_temporal ON hex_weather(world_id, q, r, s, modified DESC)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_season ON hex_weather(world_id, season)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_chunk ON hex_weather(world_id, chunk_q, chunk_r, chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_scale ON hex_weather(world_id, scale_level)")


        self.weather = db.t.hex_weather

        

        # Store table references using db.t notation
        self.users = db.t.user
        self.hexes = db.t.hex_data
        self.worlds = db.t.terrain_world
        self.watersheds = db.t.watershed_meta  # Add table reference
        self.borders = db.t.chunk_border  # Add table reference

        
        
        # Populate templates (after tables exist)
        #self.populate_templates()


    @staticmethod
    def get_db_path(custom_path=None):
        """Get database path. Tries package data, falls back to user directory."""
        if custom_path:
            return custom_path
        
        # Try package data directory first
        try:
            db_dir = resources.files('HexMagic').joinpath('data/db')
            db_path = Path(db_dir) / 'hexmagic.db'
            # Test if writable
            db_path.parent.mkdir(parents=True, exist_ok=True)
            db_path.touch(exist_ok=True)
            return str(db_path)
        except (PermissionError, OSError):
            # Fallback to user directory
            data_dir = Path.home() / '.hexmagic' / 'data'
            data_dir.mkdir(parents=True, exist_ok=True)
            return str(data_dir / 'hexmagic.db')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.db.close()

    
    @staticmethod
    def _get(obj, key):
        """Get attribute from dict or object."""
        return obj[key] if isinstance(obj, dict) else getattr(obj, key)

Should I create an index on boarders

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, you'll likely want indexes on `chunk_border` for the query patterns you'll use. Based on the structure, I'd suggest:

```python
# Find borders by source chunk
db.execute("CREATE INDEX IF NOT EXISTS idx_border_chunk ON chunk_border(world_id, chunk_q, chunk_r, chunk_s)")

# Find borders draining INTO a chunk (for propagating flow downstream)
db.execute("CREATE INDEX IF NOT EXISTS idx_border_downstream ON chunk_border(world_id, downstream_chunk_q, downstream_chunk_r, downstream_chunk_s)")
```

The first helps when loading/processing a chunk's outflow. The second helps when you need to find all inflow to a chunk from neighbors—useful for computing accumulated flow across chunk boundaries.

In [ ]:
#| export


#this might be something we don't export
@patch
def reset_database(self:GeoStorage):
    """Delete and recreate database. USE WITH CAUTION!"""
    db_path = Path(self.path)  # Convert string to Path
    
    if db_path.exists():
        db_path.unlink()

    self.createDB()



In [ ]:
#| export

@patch
def _latest_hex_subquery(self: GeoStorage, world_id: int, as_of: int = None) -> str:
    """SQL subquery for latest hex state, optionally at a point in time."""
    time_clause = f"AND modified <= {as_of}" if as_of else ""
    return f"""
        SELECT world_id, q, r, s, MAX(modified) as max_mod
        FROM hex_data
        WHERE world_id = {world_id} {time_clause}
        GROUP BY world_id, q, r, s
    """




In [ ]:
#| export
@patch
def query_hexes_latest(self: GeoStorage, world_id: int, as_of: int = None) -> list:
    """Get most recent version of each hex, optionally at a specific time."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """)
    
    # Get column names from cursor description
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    
    return rows


In [ ]:
#| export
@patch
def update_hex(self: GeoStorage, world_id: int, q: int, r: int, s: int,
               elevation: float = None, latitude: float = None, longitude: float = None,
               distance_from_coast: float = None, grid_index: int = None) -> SaveResult:
    """Append a new state for a hex (temporal update)."""
    now = int(datetime.now().timestamp())
    
    record = {
        'world_id': world_id,
        'q': q, 'r': r, 's': s,
        'modified': now
    }
    
    # Get previous state to preserve unchanged fields
    prev = self.query_hexes_in_radius(world_id, q, r, s, radius=0)
    if prev:
        prev = prev[0]
        record['grid_index'] = grid_index if grid_index is not None else GeoStorage._get(prev, 'grid_index')
        record['elevation'] = elevation if elevation is not None else GeoStorage._get(prev, 'elevation')
        record['latitude'] = latitude if latitude is not None else GeoStorage._get(prev, 'latitude')
        record['longitude'] = longitude if longitude is not None else GeoStorage._get(prev, 'longitude')
        record['distance_from_coast'] = distance_from_coast if distance_from_coast is not None else GeoStorage._get(prev, 'distance_from_coast')
    else:
        record['grid_index'] = grid_index or 0
        record['elevation'] = elevation or 0.0
        record['latitude'] = latitude
        record['longitude'] = longitude
        record['distance_from_coast'] = distance_from_coast
    
    try:
        self.hexes.insert(record)
        return SaveResult(None, 'updated', f'hex ({q},{r},{s}) at {now}')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    rows = list(self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ]).fetchall())
    
    if not rows:
        return []
    
    # Hardcode column order to avoid cursor.description issue
    cols = ['id', 'world_id', 'q', 'r', 's', 'grid_index', 'elevation', 
        'plate_id', 'latitude', 'longitude', 'distance_from_coast', 
        'watershed_id', 'modified']
    
    rows = [dict(zip(cols, row)) for row in rows]
    
    # Filter to actual hex distance
    return [row for row in rows
            if max(abs(row['q'] - center_q), abs(row['r'] - center_r), abs(row['s'] - center_s)) <= radius]


### World

In [ ]:
#| export
@patch
def save_world(self: GeoStorage, terrain: Terrain, name: str = "") -> SaveResult:
    """Save terrain to database, including plate_id if present."""
    now = int(datetime.now().timestamp())
    
    grid = terrain.hexGrid
    
    # Build extras dict
    extras = {}
    if terrain.geo:
        extras['geo'] = terrain.geo.encode() if hasattr(terrain.geo, 'encode') else str(terrain.geo)
    if terrain.climate:
        extras['climate'] = terrain.climate.encode() if hasattr(terrain.climate, 'encode') else str(terrain.climate)
    
    try:
        with self.db.conn:
            world_result = self.worlds.insert({
                'name': name or f"World_{now}",
                'hex_radius': grid.radius,
                'nrows': grid.nRows,
                'ncols': grid.nCols,
                'sea_level': terrain.seaLevel if isinstance(terrain.seaLevel, (int, float)) else 0.0,
                'elevation_delta': terrain.elevationDelta,
                'extras': json.dumps(extras),
                'created': now,
                'modified': now
            })
            
            world_id = world_result['id'] if isinstance(world_result, dict) else world_result
            
            # Check for plate_id field
            has_plates = 'plate_id' in terrain.fields
            
            for idx, hex_obj in enumerate(grid.hexes):
                pos = grid.index_to_hexposition(idx)
                
                lat, lon = None, None
                if terrain.geo and hasattr(terrain.geo, 'pixel_to_latlon'):
                    center = hex_obj.center
                    width = grid.nCols * grid.radius * 1.5 + grid.radius
                    height = grid.nRows * grid.radius * math.sqrt(3) + grid.radius
                    lat, lon = terrain.geo.pixel_to_latlon(center.x, center.y, width, height)
                
                dist_coast = None
                if 'distance_to_ocean' in terrain.fields:
                    dist_coast = float(terrain.fields['distance_to_ocean'][idx])
                
                # Include plate_id if available
                plate_id = int(terrain.fields['plate_id'][idx]) if has_plates else -1
                
                self.hexes.insert({
                    'world_id': world_id,
                    'q': pos.q, 'r': pos.r, 's': pos.s,
                    'grid_index': idx,
                    'elevation': float(terrain.elevations[idx]),
                    'plate_id': plate_id,
                    'latitude': lat,
                    'longitude': lon,
                    'distance_from_coast': dist_coast,
                    'modified': now
                })
            
            return SaveResult(world_id, 'saved', f'{len(grid.hexes)} hexes')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def load_world(self: GeoStorage, world_id: int, as_of: int = None) -> LoadResult:
    """Load terrain from database, including plate_id if present."""
    try:
        world = self.worlds[world_id]
        if not world:
            return LoadResult(None, 'not_found', f'World {world_id} not found')
        
        radius = GeoStorage._get(world, 'hex_radius')
        nrows = GeoStorage._get(world, 'nrows')
        ncols = GeoStorage._get(world, 'ncols')
        
        width = ncols * radius * 1.5 + radius
        height = nrows * radius * math.sqrt(3) + radius
        bounds = MapRect(MapCord(0, 0), MapSize(width, height))
        
        terrain = Terrain(bounds, radius=radius)
        terrain.hexGrid.nRows = nrows
        terrain.hexGrid.nCols = ncols
        terrain.hexGrid.adjustRadius(radius)
        terrain.elevations = np.zeros(len(terrain.hexGrid.hexes))
        terrain.elevationDelta = GeoStorage._get(world, 'elevation_delta')
        
        hex_rows = self.query_hexes_latest(world_id, as_of=as_of)
        
        # Check if any row has plate_id
        has_plates = any(GeoStorage._get(row, 'plate_id') is not None 
                        and GeoStorage._get(row, 'plate_id') >= 0 for row in hex_rows)
        
        if has_plates:
            terrain.fields['plate_id'] = np.full(len(terrain.elevations), -1, dtype=int)
        
        for row in hex_rows:
            idx = GeoStorage._get(row, 'grid_index')
            terrain.elevations[idx] = GeoStorage._get(row, 'elevation')
            
            # Restore plate_id
            plate_id = GeoStorage._get(row, 'plate_id')
            if has_plates and plate_id is not None:
                terrain.fields['plate_id'][idx] = plate_id
            
            dist = GeoStorage._get(row, 'distance_from_coast')
            if dist is not None:
                if 'distance_to_ocean' not in terrain.fields:
                    terrain.fields['distance_to_ocean'] = np.zeros(len(terrain.elevations))
                terrain.fields['distance_to_ocean'][idx] = dist
        
        extras_str = GeoStorage._get(world, 'extras')
        if extras_str:
            extras = json.loads(extras_str)
            # Could also restore geo/climate from extras here
        
        time_info = f" as of {as_of}" if as_of else ""
        return LoadResult(terrain, 'loaded', f'{len(hex_rows)} hexes{time_info}')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


## Chunk

In [ ]:
#| export
@patch
def save_chunk(self: GeoStorage, chunk: HexChunk, coarse_world_id: int = None) -> SaveResult:
    """Save HexChunk to database.
    
    Args:
        chunk: The HexChunk to save
        coarse_world_id: Optional parent world reference (for linking chunks to coarse map)
    """
    now = int(datetime.now().timestamp())
    p = chunk.position
    name = f"chunk_{p.q}_{p.r}_{p.s}"
    
    # Store in TerrainWorld (reusing schema)
    extras = {'coarse_world_id': coarse_world_id, 'is_chunk': True}
    
    with self.db.conn:
        world_result = self.worlds.insert({
            'name': name,
            'hex_radius': 0,  # Not used for chunks
            'nrows': chunk.rings,  # Store rings here
            'ncols': chunk.core_size,  # Store core_size here
            'extras': json.dumps(extras),
            'created': now,
            'modified': now
        })
        
        world_id = world_result['id'] if isinstance(world_result, dict) else world_result
        
        # Save hex data (core only, or all?)
        for idx in chunk.iter_core():  # Save core only
            local_pos = chunk.index_to_hexposition(idx)
            
            record = {
                'world_id': world_id,
                'q': local_pos.q,
                'r': local_pos.r, 
                's': local_pos.s,
                'grid_index': idx,
                'elevation': float(chunk.elevations[idx]),
                'plate_id': int(chunk.fields.get('plate_id', [-1])[idx]) if 'plate_id' in chunk.fields else -1,
                'modified': now
            }
            self.hexes.insert(record)
        
        return SaveResult(world_id, 'saved', f'{chunk.core_size} hexes')


In [ ]:
#| export
@patch  
def load_chunk(self: GeoStorage, chunk_ref: ChunkRef) -> LoadResult:
    """Load HexChunk from database."""
   

    p = chunk_ref.position
    name = f"chunk_{p.q}_{p.r}_{p.s}"
    
    # Find the most recent chunk by name - use id DESC since created has second precision
    cursor = self.db.execute(
        "SELECT * FROM terrain_world WHERE name = ? ORDER BY id DESC LIMIT 1",
        [name]
    )
    
    row = cursor.fetchone()
    if not row:
        return LoadResult(None, 'not_found', f'Chunk {name} not found')
    
    cols = [d[0] for d in cursor.description]
    world = dict(zip(cols, row))
    
    world_id = world['id']
    rings = world['nrows']
    core_size = world['ncols']
    
    # Create HexChunk
    chunk = HexChunk(chunk_ref.position, rings)
    chunk.core_size = core_size
    
    # Load hex data
    hex_rows = self.query_hexes_latest(world_id)
    
    for row in hex_rows:
        pos = HexPosition(row['q'], row['r'], row['s'])
        idx = chunk.hexposition_to_index(pos)
        if idx >= 0:
            chunk.elevations[idx] = row['elevation']
            if row.get('plate_id', -1) >= 0:
                if 'plate_id' not in chunk.fields:
                    chunk.add_field('plate_id', default=-1)
                chunk.fields['plate_id'][idx] = row['plate_id']
    
    return LoadResult(chunk, 'loaded', f'{len(hex_rows)} hexes')


### Region

@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ])
    
    # Convert to dicts like query_hexes_latest does
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    
    # Filter to actual hex distance
    result = []
    for row in rows:
        dq = abs(GeoStorage._get(row, 'q') - center_q)
        dr = abs(GeoStorage._get(row, 'r') - center_r)
        ds = abs(GeoStorage._get(row, 's') - center_s)
        dist = max(dq, dr, ds)
        if dist <= radius:
            result.append(row)
    
    return result


In [ ]:
#| export
@patch
def extract_region(self: GeoStorage, world_id: int,
                   center_q: int, center_r: int, center_s: int,
                   rings: int, padding: int = 1, as_of: int = None) -> LoadResult:
    """Extract a centered region as a new Terrain.
    
    Returns LoadResult(terrain, status, context)
    """
    world = self.worlds[world_id]
    if not world:
        return LoadResult(None, 'not_found', f'World {world_id} not found')
    
    radius = GeoStorage._get(world, 'hex_radius')
    
    # Query hexes in region
    hex_rows = self.query_hexes_in_radius(world_id, center_q, center_r, center_s, rings, as_of=as_of)
    if not hex_rows:
        return LoadResult(None, 'empty', 'No hexes found in region')
    
    # Create centered grid
    total_rings = rings + padding
    grid = HexGrid.centered(radius=radius, rings=total_rings, style=StyleCSS("simple"))
    
    # Build terrain
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {}
    terrain.elevationDelta = GeoStorage._get(world, 'elevation_delta')
    terrain.seaLevel = GeoStorage._get(world, 'sea_level')
    terrain.climate = None
    terrain.path = ""
    terrain.colorLevels = None
    
    # Compute GeoBounds from extracted hexes
    lats = [GeoStorage._get(r, 'latitude') for r in hex_rows if GeoStorage._get(r, 'latitude') is not None]
    lons = [GeoStorage._get(r, 'longitude') for r in hex_rows if GeoStorage._get(r, 'longitude') is not None]
    
    if lats and lons:
        terrain.geo = GeoBounds(
            lat_min=min(lats), lat_max=max(lats),
            lon_min=min(lons), lon_max=max(lons)
        )

    else:
        terrain.geo = None
    
    # Map DB rows to centered grid
    filled = 0
    
    for row in hex_rows:
        # Position relative to query center
        q = GeoStorage._get(row, 'q') - center_q
        r = GeoStorage._get(row, 'r') - center_r
        s = GeoStorage._get(row, 's') - center_s
        rel_pos = HexPosition(q, r, s)
        
        # Map to centered grid
        new_idx = grid.hexposition_to_index(rel_pos, origin_index=grid.middle)
        if 0 <= new_idx < len(grid.hexes):
            terrain.elevations[new_idx] = GeoStorage._get(row, 'elevation')
            
            dist = GeoStorage._get(row, 'distance_from_coast')
            if dist is not None:
                if 'distance_to_ocean' not in terrain.fields:
                    terrain.fields['distance_to_ocean'] = np.zeros(len(grid.hexes))
                terrain.fields['distance_to_ocean'][new_idx] = dist
            
            filled += 1
    
    return LoadResult(terrain, 'extracted', f'{filled}/{len(hex_rows)} hexes mapped')


In [ ]:
#| export
def create_region_coord_mapper(center_q: int, center_r: int, center_s: int, 
                                extracted_grid: HexGrid) -> callable:
    """Create mapper from extracted region index to original world (q,r,s).
    
    Args:
        center_q, center_r, center_s: The center point used for extraction
        extracted_grid: The centered grid from extract_region
    
    Returns:
        Function: new_idx -> (q, r, s) tuple in original world coords
    """
    def mapper(new_idx: int) -> tuple[int, int, int]:
        # Get position relative to extracted grid's center
        rel_pos = extracted_grid.index_to_hexposition(new_idx, origin_index=extracted_grid.middle)
        # Add back the world center offset
        return (center_q + rel_pos.q, center_r + rel_pos.r, center_s + rel_pos.s)
    
    return mapper


In [ ]:
#| export
@patch
def create_region_index_mapper(self: GeoStorage, world_id: int,
                                center_q: int, center_r: int, center_s: int,
                                extracted_grid: HexGrid) -> callable:
    """Create mapper from extracted region index to original world grid_index.
    
    Args:
        world_id: The source world
        center_q, center_r, center_s: The center point used for extraction
        extracted_grid: The centered grid from extract_region
    
    Returns:
        Function: new_idx -> original_grid_index (or -1 if not in world)
    """
    # Build lookup dict from DB (q,r,s) -> grid_index
    hex_rows = list(self.hexes.rows_where('world_id = ?', [world_id]))
    coord_to_idx = {
        (GeoStorage._get(r, 'q'), GeoStorage._get(r, 'r'), GeoStorage._get(r, 's')): 
        GeoStorage._get(r, 'grid_index')
        for r in hex_rows
    }
    
    def mapper(new_idx: int) -> int:
        rel_pos = extracted_grid.index_to_hexposition(new_idx, origin_index=extracted_grid.middle)
        world_coords = (center_q + rel_pos.q, center_r + rel_pos.r, center_s + rel_pos.s)
        return coord_to_idx.get(world_coords, -1)
    
    return mapper


### Weather

In [ ]:
#| export
@patch
def save_weather(self: GeoStorage, terrain: Terrain, world_id: int, 
                 season: str = "annual") -> SaveResult:
    """Save weather data for a terrain to database.
    
    Args:
        terrain: Terrain with weather fields computed
        world_id: World ID to associate weather with
        season: Season identifier (e.g., "annual", "summer", "winter")
    
    Returns:
        SaveResult(count, status, context)
    """
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    
    # Check for required weather fields
    if 'temperature' not in terrain.fields or 'precipitation' not in terrain.fields:
        return SaveResult(None, 'error', 'Missing temperature or precipitation fields')
    
    try:
        with self.db.conn:
            records = []
            for idx in range(len(terrain.elevations)):
                pos = grid.index_to_hexposition(idx)
                
                record = {
                    'world_id': world_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'temperature': float(terrain.fields['temperature'][idx]),
                    'precipitation': float(terrain.fields['precipitation'][idx]),
                    'humidity': float(terrain.fields['humidity'][idx]) if 'humidity' in terrain.fields else None,
                    'climate_pet': float(terrain.fields['climate_pet'][idx]) if 'climate_pet' in terrain.fields else None,
                    'aridity_index': float(terrain.fields['aridity_index'][idx]) if 'aridity_index' in terrain.fields else None,
                    'temp_seasonality': float(terrain.fields['temp_seasonality'][idx]) if 'temp_seasonality' in terrain.fields else None,
                    'precip_seasonality': float(terrain.fields['precip_seasonality'][idx]) if 'precip_seasonality' in terrain.fields else None,
                    'distance_from_coast': float(terrain.fields['distance_to_coast'][idx]) if 'distance_to_coast' in terrain.fields else None,
                    'season': season,
                    'modified': now
                }
                records.append(record)
            
            for record in records:
                self.weather.insert(record)
            
            return SaveResult(len(records), 'saved', f'{len(records)} weather records for season={season}')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))




In [ ]:
#| export

@patch
def load_weather(self: GeoStorage, world_id: int, terrain: Terrain,
                 season: str = "annual", as_of: int = None) -> LoadResult:
    """Load weather data into a terrain's fields.
    
    Args:
        world_id: World ID to load weather for
        terrain: Terrain to populate with weather fields
        season: Season to load
        as_of: Optional timestamp for temporal query
    
    Returns:
        LoadResult(terrain, status, context)
    """
    try:
        # Build temporal subquery
        time_clause = f"AND modified <= {as_of}" if as_of else ""
        
        cursor = self.db.execute(f"""
            SELECT hw.* FROM hex_weather hw
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_weather
                WHERE world_id = ? AND season = ? {time_clause}
                GROUP BY world_id, q, r, s
            ) latest 
                ON hw.world_id = latest.world_id 
                AND hw.q = latest.q AND hw.r = latest.r AND hw.s = latest.s
                AND hw.modified = latest.max_mod
            WHERE hw.season = ?
        """, [world_id, season, season])
        
        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
        
        if not rows:
            return LoadResult(terrain, 'empty', f'No weather data for world={world_id}, season={season}')
        
        # Initialize field arrays
        n_hexes = len(terrain.elevations)
        terrain.fields['temperature'] = np.zeros(n_hexes)
        terrain.fields['precipitation'] = np.zeros(n_hexes)
        terrain.fields['humidity'] = np.zeros(n_hexes)
        terrain.fields['climate_pet'] = np.zeros(n_hexes)
        terrain.fields['aridity_index'] = np.zeros(n_hexes)
        terrain.fields['temp_seasonality'] = np.zeros(n_hexes)
        terrain.fields['precip_seasonality'] = np.zeros(n_hexes)
        terrain.fields['distance_to_coast'] = np.zeros(n_hexes)
        
        # Map weather data to terrain indices
        grid = terrain.hexGrid
        filled = 0
        
        for row in rows:
            pos = HexPosition(row['q'], row['r'], row['s'])
            idx = grid.hexposition_to_index(pos)
            
            if 0 <= idx < n_hexes:
                terrain.fields['temperature'][idx] = row['temperature'] or 0.0
                terrain.fields['precipitation'][idx] = row['precipitation'] or 0.0
                if row['humidity'] is not None:
                    terrain.fields['humidity'][idx] = row['humidity']
                if row['climate_pet'] is not None:
                    terrain.fields['climate_pet'][idx] = row['climate_pet']
                if row['aridity_index'] is not None:
                    terrain.fields['aridity_index'][idx] = row['aridity_index']
                if row['temp_seasonality'] is not None:
                    terrain.fields['temp_seasonality'][idx] = row['temp_seasonality']
                if row['precip_seasonality'] is not None:
                    terrain.fields['precip_seasonality'][idx] = row['precip_seasonality']
                if row['distance_from_coast'] is not None:
                    terrain.fields['distance_to_coast'][idx] = row['distance_from_coast']
                filled += 1
        
        time_info = f" as of {as_of}" if as_of else ""
        return LoadResult(terrain, 'loaded', f'{filled} weather records for season={season}{time_info}')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def load_chunk_weather(self: GeoStorage, world_id: int,
                       chunk: HexChunk, chunk_ref: ChunkRef,
                       season: str = "annual") -> LoadResult:
    """Load weather for a specific chunk from database.
    
    Args:
        world_id: World ID
        chunk: HexChunk to populate with weather fields
        chunk_ref: ChunkRef identifying the chunk
        season: Season to load
    
    Returns:
        LoadResult(chunk, status, context)
    """
    try:
        p = chunk_ref.position
        
        cursor = self.db.execute("""
            SELECT hw.* FROM hex_weather hw
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_weather
                WHERE world_id = ? AND season = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                GROUP BY world_id, q, r, s
            ) latest 
                ON hw.world_id = latest.world_id 
                AND hw.q = latest.q AND hw.r = latest.r AND hw.s = latest.s
                AND hw.modified = latest.max_mod
            WHERE hw.season = ?
        """, [world_id, season, p.q, p.r, p.s, season])
        
        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
        
        if not rows:
            return LoadResult(chunk, 'not_found', 
                            f'No weather for chunk ({p.q},{p.r},{p.s}), season={season}')
        
        # Initialize fields if needed
        if 'temperature' not in chunk.fields:
            chunk.add_field('temperature', default=15.0)
        if 'precipitation' not in chunk.fields:
            chunk.add_field('precipitation', default=500.0)
        
        filled = 0
        for row in rows:
            # Convert world coords to local chunk position
            world_pos = HexPosition(row['q'], row['r'], row['s'])
            idx = chunk.world_to_index(world_pos)
            
            if idx >= 0:
                chunk.fields['temperature'][idx] = row['temperature'] or 15.0
                chunk.fields['precipitation'][idx] = row['precipitation'] or 500.0
                filled += 1
        
        return LoadResult(chunk, 'loaded', f'{filled} weather records for chunk ({p.q},{p.r},{p.s})')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


@patch
def has_chunk_weather(self: GeoStorage, world_id: int, 
                      chunk_ref: ChunkRef, 
                      season: str = "annual",
                      climate_name: str = None,
                      min_count: int = 1) -> bool:
    """Check if valid cached weather exists for chunk.
    
    Args:
        world_id: World ID
        chunk_ref: ChunkRef identifying the chunk
        season: Season to check
        climate_name: If provided, verify weather was computed with this climate
        min_count: Minimum number of weather records to consider valid
    
    Returns:
        True if valid cached weather exists
    """
    try:
        p = chunk_ref.position
        
        # Build query
        if climate_name:
            cursor = self.db.execute("""
                SELECT COUNT(*) FROM hex_weather
                WHERE world_id = ? AND season = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND climate_name = ?
            """, [world_id, season, p.q, p.r, p.s, climate_name])
        else:
            cursor = self.db.execute("""
                SELECT COUNT(*) FROM hex_weather
                WHERE world_id = ? AND season = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
            """, [world_id, season, p.q, p.r, p.s])
        
        count = cursor.fetchone()[0]
        return count >= min_count
    
    except Exception:
        return False


@patch
def save_chunk_weather(self: GeoStorage, chunk: HexChunk, 
                       chunk_ref: ChunkRef, world_id: int,
                       climate_name: str = "",
                       wind_dir: float = None,
                       season: str = "annual") -> SaveResult:
    """Save weather for a single chunk.
    
    Args:
        chunk: HexChunk with computed weather fields
        chunk_ref: ChunkRef identifying the chunk
        world_id: World ID
        climate_name: Climate preset name (for cache invalidation)
        wind_dir: Wind direction used in computation
        season: Season identifier
    
    Returns:
        SaveResult(count, status, context)
    """
    now = int(datetime.now().timestamp())
    p = chunk_ref.position
    
    if 'temperature' not in chunk.fields or 'precipitation' not in chunk.fields:
        return SaveResult(None, 'error', 'Chunk missing weather fields')
    
    try:
        with self.db.conn:
            count = 0
            for idx in chunk.iter_core():  # Save core only
                world_pos = chunk.index_to_world(idx)
                
                record = {
                    'world_id': world_id,
                    'q': world_pos.q,
                    'r': world_pos.r,
                    's': world_pos.s,
                    'chunk_q': p.q,
                    'chunk_r': p.r,
                    'chunk_s': p.s,
                    'temperature': float(chunk.fields['temperature'][idx]),
                    'precipitation': float(chunk.fields['precipitation'][idx]),
                    'climate_name': climate_name,
                    'wind_dir': wind_dir,
                    'season': season,
                    'scale_level': 1,  # Fine/chunk level
                    'modified': now
                }
                self.weather.insert(record)
                count += 1
            
            return SaveResult(count, 'saved', 
                            f'{count} weather records for chunk ({p.q},{p.r},{p.s})')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


### watershed

In [ ]:
#| export
@patch
def save_watersheds(self: GeoStorage, terrain: Terrain, 
                    watersheds: list, world_id: int) -> SaveResult:
    """Save watersheds and assign watershed_id to hexes.
    
    Args:
        terrain: Terrain with computed watersheds
        watersheds: List of Watershed objects
        world_id: World to associate with
    
    Returns:
        SaveResult(count, status, context)
    """
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    
    try:
        with self.db.conn:
            
            
            # Build hex -> watershed mapping
            hex_to_watershed = {}
            
            for i, watershed in enumerate(watersheds):
                # Get terminal position
                terminal_idx = watershed.terminal_hex
                if terminal_idx is not None:
                    terminal_pos = grid.index_to_hexposition(terminal_idx)
                else:
                    terminal_pos = HexPosition(0, 0, 0)
                
                # Calculate stats
                flows = watershed.calculate_flow()
                max_flow = max(flows.values()) if flows else 0
                
                # Save watershed metadata
                ws_record = {
                    'world_id': world_id,
                    'name': f"watershed_{i}",
                    'terminal_q': terminal_pos.q,
                    'terminal_r': terminal_pos.r,
                    'terminal_s': terminal_pos.s,
                    'is_ocean': watershed.is_ocean,
                    'total_flow': max_flow,
                    'area_hexes': len(watershed.region.hexes),
                    'river_tree': watershed.tributary.encode(),
                    'style': watershed.style.encode() if hasattr(watershed.style, 'encode') else str(watershed.style),
                    'created': now,
                    'modified': now
                }
                
                result = self.watersheds.insert(ws_record)
                ws_id = result['id'] if isinstance(result, dict) else result
                
                # Map hexes to this watershed
                for hex_idx in watershed.region.hexes:
                    hex_to_watershed[hex_idx] = ws_id
            
            # Update hex_data with watershed_id
            for hex_idx, ws_id in hex_to_watershed.items():
                pos = grid.index_to_hexposition(hex_idx)
                
                # Insert new record with watershed_id
                self.hexes.insert({
                    'world_id': world_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'grid_index': hex_idx,
                    'elevation': float(terrain.elevations[hex_idx]),
                    'watershed_id': ws_id,
                    'modified': now
                })
            
            return SaveResult(len(watersheds), 'saved', 
                            f'{len(watersheds)} watersheds, {len(hex_to_watershed)} hexes assigned')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))





@patch
def query_watershed_hexes(self: GeoStorage, world_id: int, 
                          watershed_id: int) -> list:
    """Get all hexes belonging to a specific watershed."""
    cursor = self.db.execute("""
        SELECT h.* FROM hex_data h
        INNER JOIN (
            SELECT world_id, q, r, s, MAX(modified) as max_mod
            FROM hex_data
            WHERE world_id = ? AND watershed_id = ?
            GROUP BY world_id, q, r, s
        ) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.watershed_id = ?
    """, [world_id, watershed_id, watershed_id])
    
    cols = [d[0] for d in cursor.description]
    return [dict(zip(cols, row)) for row in cursor.fetchall()]


In [ ]:
#| export
@patch
def load_watersheds(self: GeoStorage, world_id: int, 
                    terrain: Terrain) -> LoadResult:
    """Load watersheds for a world.
    
    Args:
        world_id: World to load from
        terrain: Terrain to attach watersheds to
    
    Returns:
        LoadResult(watersheds_list, status, context)
    """
    try:
        # Query watershed metadata
        ws_rows = list(self.watersheds.rows_where(
    'world_id = ?', [world_id]
))
        
        
        if not ws_rows:
            return LoadResult([], 'empty', f'No watersheds for world {world_id}')
        
        # Query hex -> watershed mapping
        hex_rows = self.query_hexes_latest(world_id)
        
        # Group hexes by watershed_id
        ws_hexes = {}  # watershed_id -> set of hex indices
        for row in hex_rows:
            ws_id = row.get('watershed_id')
            if ws_id is not None:
                if ws_id not in ws_hexes:
                    ws_hexes[ws_id] = set()
                ws_hexes[ws_id].add(row['grid_index'])
        
        # Reconstruct Watershed objects
        watersheds = []
        for ws_row in ws_rows:
            ws_id = ws_row['id']
            
            
            # Get hexes for this watershed
            hexes = ws_hexes.get(ws_id, set())
            region = HexRegion(hexes=hexes, hexGrid=terrain.hexGrid)
            
            # Decode river
            river_str = ws_row['river_tree']
            river = River.decode(river_str, terrain) if river_str else River(terrain)
            
            # Decode style
            style_str = ws_row['style']
            if style_str:
                style = StyleCSS.decode(style_str)
            else:
                style = StyleCSS(f"ws_{ws_id}", fill="#cccccc")
            
            watershed = Watershed(
                region=region,
                tributary=river,
                style=style
            )
            watersheds.append(watershed)
        
        return LoadResult(watersheds, 'loaded', f'{len(watersheds)} watersheds')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))

I got the following error. I am trying to make plate id persisenct can you give me better load and save world



Assistant: 418

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right. The issue is that load_chunk returns terrain from storage, but save_world/load_world likely doesn't preserve the fields dict (including plate_id). So when we load a previously saved chunk, fields is empty or missing plate_id.

Quick fix - always generate fresh in the pathfinding test (skip load):
Copied!

# In hpa_pathfind_v2, replace:
terrain = gen.load_chunk(chunk_ref)
if terrain is None:
    terrain = gen.generate_chunk(chunk_ref)
    gen.save_chunk(terrain, chunk_ref)

# With just:
terrain = gen.generate_chunk(chunk_ref)

Proper fix - ensure GeoStorage.save_world and load_world round-trip the fields dict. Check if save_world stores fields like:
Copied!

# In save_world, after saving elevations:
for field_name, field_data in terrain.fields.items():
    for idx, value in enumerate(field_data):
        # Store in hex row or separate fields table
        ...

And in load_world:
Copied!

# Reconstruct fields from stored data
terrain.fields = {}
for field_name in stored_field_names:
    terrain.fields[field_name] = np.array([...])

What does your current save_world/load_world do with terrain.fields?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The issue is that `save_world` saves `plate_id` to the DB, but `load_world` doesn't restore it to `terrain.fields`. Here's the fix:

```python
@patch
def load_world(self: GeoStorage, world_id: int, as_of: int = None) -> LoadResult:
    """Load terrain from database, including plate_id if present."""
    try:
        world = self.worlds[world_id]
        if not world:
            return LoadResult(None, 'not_found', f'World {world_id} not found')
        
        radius = GeoStorage._get(world, 'hex_radius')
        nrows = GeoStorage._get(world, 'nrows')
        ncols = GeoStorage._get(world, 'ncols')
        
        width = ncols * radius * 1.5 + radius
        height = nrows * radius * math.sqrt(3) + radius
        bounds = MapRect(MapCord(0, 0), MapSize(width, height))
        
        terrain = Terrain(bounds, radius=radius)
        terrain.hexGrid.nRows = nrows
        terrain.hexGrid.nCols = ncols
        terrain.hexGrid.adjustRadius(radius)
        terrain.elevations = np.zeros(len(terrain.hexGrid.hexes))
        terrain.elevationDelta = GeoStorage._get(world, 'elevation_delta')
        
        hex_rows = self.query_hexes_latest(world_id, as_of=as_of)
        
        # Check if any row has plate_id
        has_plates = any(GeoStorage._get(row, 'plate_id') is not None 
                        and GeoStorage._get(row, 'plate_id') >= 0 for row in hex_rows)
        
        if has_plates:
            terrain.fields['plate_id'] = np.full(len(terrain.elevations), -1, dtype=int)
        
        for row in hex_rows:
            idx = GeoStorage._get(row, 'grid_index')
            terrain.elevations[idx] = GeoStorage._get(row, 'elevation')
            
            # Restore plate_id
            plate_id = GeoStorage._get(row, 'plate_id')
            if has_plates and plate_id is not None:
                terrain.fields['plate_id'][idx] = plate_id
            
            dist = GeoStorage._get(row, 'distance_from_coast')
            if dist is not None:
                if 'distance_to_ocean' not in terrain.fields:
                    terrain.fields['distance_to_ocean'] = np.zeros(len(terrain.elevations))
                terrain.fields['distance_to_ocean'][idx] = dist
        
        extras_str = GeoStorage._get(world, 'extras')
        if extras_str:
            extras = json.loads(extras_str)
            # Could also restore geo/climate from extras here
        
        time_info = f" as of {as_of}" if as_of else ""
        return LoadResult(terrain, 'loaded', f'{len(hex_rows)} hexes{time_info}')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))
```

Also need to update `query_hexes_in_radius` to include `plate_id` in the column list:

```python
# In query_hexes_in_radius, update the cols list:
cols = ['id', 'world_id', 'q', 'r', 's', 'grid_index', 'elevation', 'plate_id',
        'latitude', 'longitude', 'distance_from_coast', 'watershed_id', 'modified']
```

In [ ]:
#| export
# === WATERSHED PERSISTENCE ===

@patch
def save_chunk_watersheds(self: GeoStorage, chunk: HexChunk, 
                          chunk_ref: ChunkRef, world_id: int) -> SaveResult:
    """Save watershed assignments for a chunk."""
    now = int(datetime.now().timestamp())
    p = chunk_ref.position
    
    if 'watershed_id' not in chunk.fields:
        return SaveResult(None, 'error', 'Chunk missing watershed_id field')
    
    try:
        with self.db.conn:
            count = 0
            for idx in chunk.iter_core():
                world_pos = chunk.index_to_world(idx)
                ws_id = int(chunk.fields['watershed_id'][idx])
                
                if ws_id < 0:
                    continue  # Skip unassigned/ocean
                
                # Update hex_data with watershed_id
                self.hexes.insert({
                    'world_id': world_id,
                    'q': world_pos.q,
                    'r': world_pos.r,
                    's': world_pos.s,
                    'grid_index': idx,
                    'elevation': float(chunk.elevations[idx]),
                    'watershed_id': ws_id,
                    'modified': now
                })
                count += 1
            
            return SaveResult(count, 'saved', 
                            f'{count} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_chunk_watersheds(self: GeoStorage, world_id: int,
                          chunk: HexChunk, chunk_ref: ChunkRef) -> LoadResult:
    """Load watershed assignments for a chunk."""
    try:
        p = chunk_ref.position
        
        # Query hexes with watershed_id for this chunk's world positions
        hex_rows = []
        for idx in chunk.iter_core():
            world_pos = chunk.index_to_world(idx)
            rows = self.query_hexes_in_radius(world_id, world_pos.q, world_pos.r, world_pos.s, radius=0)
            if rows:
                hex_rows.append((idx, rows[0]))
        
        if not hex_rows:
            return LoadResult(chunk, 'not_found', 
                            f'No watershed data for chunk ({p.q},{p.r},{p.s})')
        
        # Initialize field if needed
        if 'watershed_id' not in chunk.fields:
            chunk.add_field('watershed_id', default=-1)
        
        filled = 0
        for idx, row in hex_rows:
            ws_id = row.get('watershed_id')
            if ws_id is not None and ws_id >= 0:
                chunk.fields['watershed_id'][idx] = ws_id
                filled += 1
        
        return LoadResult(chunk, 'loaded', 
                         f'{filled} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def has_chunk_watersheds(self: GeoStorage, world_id: int, 
                         chunk_ref: ChunkRef,
                         min_count: int = 1) -> bool:
    """Check if watershed data exists for chunk."""
    try:
        center = chunk_ref.center_hex
        # Use larger radius to cover more of the chunk
        radius = chunk_ref.rings  # Full chunk radius, not just 3
        
        rows = self.query_hexes_in_radius(world_id, center.q, center.r, center.s, radius=radius)
        
        ws_count = sum(1 for r in rows if r.get('watershed_id') is not None and r.get('watershed_id') >= 0)
        return ws_count >= min_count
    except Exception:
        return False



## Debugger

In [ ]:
#| export
class GeoStorageDebugger:
    """Test harness for HexServer with automatic cleanup."""
    
    def __init__(self, keep_on_error=False):
        """
        Args:
            keep_on_error: If True, preserve database when tests fail
        """
        self.temp_dir = tempfile.mkdtemp(prefix='hexmagic_test_')
        self.db_path = os.path.join(self.temp_dir, 'test.db')
        self.server = GeoStorage(custom_path=self.db_path)
        self.keep_on_error = keep_on_error
        self.failed = False
        
    def preserve_db(self, test_name):
        """Copy database to inspection directory."""
        inspect_dir = Path.home() / '.hexmagic' / 'test_failures'
        inspect_dir.mkdir(parents=True, exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dest = inspect_dir / f"{test_name}_{timestamp}.db"
        
        shutil.copy2(self.db_path, dest)
        print(f"⚠ Database preserved at: {dest}")
        
    def close(self):
        """Clean up test database."""
        self.server.db.close()
        
        if not self.failed or not self.keep_on_error:
            shutil.rmtree(self.temp_dir)
            print("✓ Test database cleaned up")
        else:
            print(f"⚠ Test database kept at: {self.temp_dir}")
            
    def __enter__(self):
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.failed = True
        self.close()


In [ ]:
#| export
@patch
def test_save_load_terrain(self: GeoStorageDebugger):
    """Test saving and loading a terrain."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save
    save_result = self.server.save_world(terrain, name="Australia Test")
    assert save_result.status == 'saved', f"Save failed: {save_result.status} - {save_result.context}"
    print(f"✓ Saved world {save_result.id}: {save_result.context}")
    
    # Load
    load_result = self.server.load_world(save_result.id)
    assert load_result.status == 'loaded', f"Load failed: {load_result.status} - {load_result.context}"
    print(f"✓ Loaded world: {load_result.context}")
    
    # Verify
    loaded = load_result.data
    assert len(loaded.elevations) == len(terrain.elevations), "Elevation count mismatch"
    assert np.allclose(loaded.elevations, terrain.elevations), "Elevation values don't match"
    print(f"✓ Elevations match ({len(loaded.elevations)} hexes)")


In [ ]:
#| export
@patch
def test_query_radius(self: GeoStorageDebugger):
    """Test querying hexes by position and radius."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save
    save_result = self.server.save_world(terrain, name="Query Test")
    assert save_result.status == 'saved'
    
    # Query center region (q=0, r=0, s=0 with radius 5)
    hexes = self.server.query_hexes_in_radius(save_result.id, 0, 0, 0, radius=5)
    
    # Should get some hexes (exact count depends on map)
    print(f"✓ Found {len(hexes)} hexes within radius 5 of origin")
    assert len(hexes) > 0, "Expected some hexes near origin"


In [ ]:
#| export
@patch
def test_extract_region(self: GeoStorageDebugger):
    """Test extracting a centered region."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save full terrain
    save_result = self.server.save_world(terrain, name="Extract Test")
    assert save_result.status == 'saved'
    
    # Extract a region around origin
    result = self.server.extract_region(save_result.id, 0, 0, 0, rings=10)
    assert result.status == 'extracted', f"Extract failed: {result.context}"
    
    extracted = result.data
    print(f"✓ Extracted region: {result.context}")
    print(f"  Grid size: {len(extracted.hexGrid.hexes)} hexes")
    
    # Verify it's centered
    assert extracted.hexGrid.middle is not None
    print(f"  Center index: {extracted.hexGrid.middle}")


In [ ]:
#| export
@patch
def test_region_mappers(self: GeoStorageDebugger):
    """Test that region mappers correctly translate indices."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    save_result = self.server.save_world(terrain, name="Mapper Test")
    assert save_result.status == 'saved'
    
    center_q, center_r, center_s = 5, -3, -2
    result = self.server.extract_region(save_result.id, center_q, center_r, center_s, rings=5)
    assert result.status == 'extracted'
    
    extracted = result.data
    grid = extracted.hexGrid
    
    # Test coord mapper
    coord_mapper = create_region_coord_mapper(center_q, center_r, center_s, grid)
    
    # Center of extracted grid should map back to query center
    center_coords = coord_mapper(grid.middle)
    assert center_coords == (center_q, center_r, center_s), f"Center mismatch: {center_coords}"
    print(f"✓ Coord mapper: center maps to {center_coords}")
    
    # Test index mapper  
    idx_mapper = self.server.create_region_index_mapper(
        save_result.id, center_q, center_r, center_s, grid
    )
    
    # Verify some mappings
    valid_count = sum(1 for i in range(len(grid.hexes)) if idx_mapper(i) >= 0)
    print(f"✓ Index mapper: {valid_count}/{len(grid.hexes)} indices map to original")


In [ ]:
#| export
@patch
def test_save_load_weather(self: GeoStorageDebugger):
    """Test saving and loading weather data."""
    # Use sfTerr which already has weather fields computed
    with open('sf_coast_weather.txt') as f:
        terrain = Terrain.decode(f.read())
    
    # First save the world (need world_id for weather)
    save_world_result = self.server.save_world(terrain, name="SF Weather Test")
    assert save_world_result.status == 'saved', f"World save failed: {save_world_result.context}"
    world_id = save_world_result.id
    print(f"✓ Saved world {world_id}")
    
    # Save weather
    save_weather_result = self.server.save_weather(terrain, world_id, season="annual")
    assert save_weather_result.status == 'saved', f"Weather save failed: {save_weather_result.context}"
    print(f"✓ Saved weather: {save_weather_result.context}")
    
    # Load world (without weather)
    load_world_result = self.server.load_world(world_id)
    assert load_world_result.status == 'loaded'
    loaded_terrain = load_world_result.data
    
    # Verify no weather fields yet
    assert 'temperature' not in loaded_terrain.fields or loaded_terrain.fields['temperature'].sum() == 0
    print(f"✓ Loaded world without weather fields")
    
    # Load weather into terrain
    load_weather_result = self.server.load_weather(world_id, loaded_terrain, season="annual")
    assert load_weather_result.status == 'loaded', f"Weather load failed: {load_weather_result.context}"
    print(f"✓ Loaded weather: {load_weather_result.context}")
    
    # Verify weather data matches
    assert np.allclose(loaded_terrain.fields['temperature'], terrain.fields['temperature']), "Temperature mismatch"
    assert np.allclose(loaded_terrain.fields['precipitation'], terrain.fields['precipitation']), "Precipitation mismatch"
    print(f"✓ Weather data matches original")
    
    # Test seasonal storage - save summer data with modified temps
    terrain.fields['temperature'] = terrain.fields['temperature'] + 5  # Summer is warmer
    save_summer = self.server.save_weather(terrain, world_id, season="summer")
    assert save_summer.status == 'saved'
    print(f"✓ Saved summer weather")
    
    # Load summer and verify it's different from annual
    load_summer = self.server.load_weather(world_id, loaded_terrain, season="summer")
    assert load_summer.status == 'loaded'
    assert loaded_terrain.fields['temperature'].mean() > terrain.fields['temperature'].mean() - 5 + 4  # Should be ~5°C warmer
    print(f"✓ Summer temperatures correctly loaded (warmer than annual)")


#| export
@patch
def test_chunked_terrain_generator(self: GeoStorageDebugger):
    """Test chunked terrain generation pipeline."""
    generator = ChunkedTerrainGenerator(
        storage=self.server,
        coarse_radius=50,      # Large hexes for overview
        fine_radius=10,        # Detailed hexes for chunks
        chunk_rings=8,         # Small chunks for testing
        halo_rings=2
    )
    
    # Phase 1: Generate coarse map
    bounds = MapRect(MapCord(0, 0), MapSize(800, 600))
    coarse = generator.generate_coarse(
        bounds,
        num_plates=5,
        ocean_fraction=0.3,
        name="chunked_test_coarse"
    )
    
    assert generator.coarse_world_id is not None, "Coarse world not saved"
    assert len(coarse.elevations) > 0, "Coarse terrain has no hexes"
    print(f"✓ Generated coarse map: {len(coarse.hexGrid.hexes)} hexes, world_id={generator.coarse_world_id}")
    
    # Phase 2: Generate a single chunk at origin
    origin_chunk = ChunkRef(HexPosition(0, 0, 0), generator.chunk_rings)
    chunk_terrain = generator.generate_chunk(origin_chunk, coarse)
    
    assert len(chunk_terrain.elevations) > 0, "Chunk terrain empty"
    assert generator.scale == 5.0, f"Expected scale 5.0, got {generator.scale}"
    print(f"✓ Generated chunk at origin: {len(chunk_terrain.hexGrid.hexes)} hexes")
    
    # Phase 3: Save chunk
    chunk_id = generator.save_chunk(chunk_terrain, origin_chunk)
    assert chunk_id is not None, "Chunk not saved"
    assert origin_chunk.key in generator.chunk_world_ids
    print(f"✓ Saved chunk: world_id={chunk_id}")
    
    # Phase 4: Load chunk back
    loaded_chunk = generator.load_chunk(origin_chunk)
    assert loaded_chunk is not None, "Failed to load chunk"
    assert len(loaded_chunk.elevations) > 0
    print(f"✓ Loaded chunk: {len(loaded_chunk.hexGrid.hexes)} hexes")
    
    # Test coordinate mapping
    world_pos = HexPosition(10, -5, -5)
    chunk_ref = generator.world_to_chunk_ref(world_pos)
    print(f"✓ World pos {world_pos} maps to chunk {chunk_ref.position}")
    
    # Test chunk iteration
    chunks = list(generator.chunk_refs(max_ring=1))
    assert len(chunks) == 7, f"Expected 7 chunks (1 center + 6 ring), got {len(chunks)}"
    print(f"✓ Chunk iteration: {len(chunks)} chunks in ring 0-1")


In [ ]:
#| export
@patch
def test_save_load_watersheds(self: GeoStorageDebugger):
    """Test watershed save/load."""
    # Load SF terrain with weather
    with open('sf_coast_weather.txt') as f:
        terrain = Terrain.decode(f.read())
        terrain.climate = TerrainPatterns(terrain).weatherPatterns()["mediterranean"]
    
    # Compute watersheds
    basins = DrainageBasins(terrain)
    
    # Save world first
    save_world = self.server.save_world(terrain, name="Watershed Test")
    assert save_world.status == 'saved'
    world_id = save_world.id
    print(f"✓ Saved world {world_id}")
    
    # Save watersheds
    save_ws = self.server.save_watersheds(terrain, basins.sheds, world_id)
    assert save_ws.status == 'saved', f"Failed: {save_ws.context}"
    print(f"✓ Saved watersheds: {save_ws.context}")
    
    # Load world (fresh terrain)
    load_world = self.server.load_world(world_id)
    loaded_terrain = load_world.data
    
    # Load watersheds
    load_ws = self.server.load_watersheds(world_id, loaded_terrain)
    assert load_ws.status == 'loaded', f"Failed: {load_ws.context}"
    print(f"✓ Loaded watersheds: {load_ws.context}")
    
    # Verify counts match
    assert len(load_ws.data) == len(basins.sheds), "Watershed count mismatch"
    print(f"✓ Watershed count matches: {len(load_ws.data)}")
    
    # Test query by watershed
    if load_ws.data:
        first_ws = load_ws.data[0]
        # Get the watershed_id from the DB
        ws_rows = list(self.server.db.t.watershed_meta.rows_where(
            'world_id = ?', [world_id]
        ))
        if ws_rows:
            hexes = self.server.query_watershed_hexes(world_id, ws_rows[0]['id'])
            print(f"✓ Query watershed hexes: {len(hexes)} hexes in watershed 0")


#| export
@patch
def test_smallWs(self: GeoStorageDebugger):
    """Generate a small world with a single watershed."""
    # Set up the chunked generator
    generator = ChunkedTerrainGenerator(
        storage=self.server,
        coarse_radius=100,     # Large hexes for overview (~40-60 visible)
        fine_radius=10,        # Detailed hexes for chunks
        chunk_rings=15,        # Rings per chunk
        halo_rings=3
    )

    # Generate the coarse world overview
    bounds = MapRect(MapCord(0, 0), MapSize(4000, 3000))
    coarse_world = generator.generate_coarse(
        bounds,
        num_plates=12,
        ocean_fraction=0.5,
        oceanic_sides=['W', 'E'],  # Oceans on west and east
        name="demo_coarse",
        max_lakes=3
    )

    print(f"Coarse world: {len(coarse_world.hexGrid.hexes)} hexes, world_id={generator.coarse_world_id}")

    # Generate a detailed chunk at the origin (or pick an interesting spot)
    origin_chunk = generator.world_to_chunk_ref(HexPosition(0, 0, 0))
    detail_terrain = generator.generate_chunk(origin_chunk, coarse_world)

    # Add weather to the detail chunk
    generator.compute_chunk_weather(detail_terrain)

    # Save the chunk
    chunk_id = generator.save_chunk(detail_terrain, origin_chunk)
    print("Testing large world")
    coarse_world.hexGrid.adjustRadius(15)
    coarse_world.colorMap()
    coarse_world.hexGrid.update()
    print("Testing small world")
    detail_terrain.hexGrid.adjustRadius(15)
    detail_terrain.colorLevels = coarse_world.colorLevels
    detail_terrain.seaLevel = coarse_world.seaLevel
    detail_terrain.colorMap()
    print("validating detail")
    print(type(detail_terrain.hexGrid.hexes[0].style))
    for i, hex in enumerate(detail_terrain.hexGrid.hexes):
        if not hasattr(hex.style, 'name'):
            print(f"Hex {i} has bad style: {hex.style} (type: {type(hex.style)})")
            break

    print(detail_terrain.hexGrid.hexes[0].style)
    detail_terrain.hexGrid.update()
    print(f"Detail chunk: {len(detail_terrain.hexGrid.hexes)} hexes, world_id={chunk_id}")

In [ ]:
#| export
@patch
def test_multiple_worlds(self: GeoStorageDebugger):
    """Test saving and loading multiple independent worlds."""
    td = TerraDemo()
    
    # Create two different terrains
    terrain_a = td.aussie_map()
    terrain_b = td.aussie_map()  # Different instance
    terrain_b.elevations = terrain_b.elevations * 1.5  # Make it different
    
    # Save both
    result_a = self.server.save_world(terrain_a, name="World A")
    assert result_a.status == 'saved', f"World A save failed: {result_a.context}"
    assert result_a.id == 1, f"Expected id=1, got {result_a.id}"
    print(f"✓ Saved World A: id={result_a.id}")
    
    result_b = self.server.save_world(terrain_b, name="World B")
    assert result_b.status == 'saved', f"World B save failed: {result_b.context}"
    assert result_b.id == 2, f"Expected id=2, got {result_b.id}"
    print(f"✓ Saved World B: id={result_b.id}")
    
    # Load both back
    loaded_a = self.server.load_world(result_a.id)
    loaded_b = self.server.load_world(result_b.id)
    
    assert loaded_a.status == 'loaded', f"World A load failed"
    assert loaded_b.status == 'loaded', f"World B load failed"
    
    # Verify data is distinct
    assert not np.allclose(loaded_a.data.elevations, loaded_b.data.elevations), \
        "Worlds should have different elevations"
    print(f"✓ Loaded both worlds with distinct data")
    
    # Verify hex queries are isolated
    hexes_a = self.server.query_hexes_in_radius(result_a.id, 0, 0, 0, radius=3)
    hexes_b = self.server.query_hexes_in_radius(result_b.id, 0, 0, 0, radius=3)
    print(f"✓ World A has {len(hexes_a)} hexes near origin, World B has {len(hexes_b)}")


#| export
@patch
def test_multiscale_watersheds(self: GeoStorageDebugger):
    """Test watershed computation at coarse and fine scales."""
    
    # Set up chunked generator
    generator = ChunkedTerrainGenerator(
        storage=self.server,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=15,
        halo_rings=3
    )
    
    # Generate coarse world with weather
    bounds = MapRect(MapCord(0, 0), MapSize(1200, 900))
    coarse = generator.generate_coarse(
        bounds,
        num_plates=8,
        ocean_fraction=0.4,
        oceanic_sides=['W'],
        name="watershed_scale_test"
    )
    
    # Compute weather on coarse
    coarse.compute_distance_from_coast()
    coarse.compute_weather()
    
    print(f"Coarse map: {len(coarse.hexGrid.hexes)} hexes")
    print(f"  Precip range: {coarse.fields['precipitation'].min():.0f} - {coarse.fields['precipitation'].max():.0f} mm")
    
    # Compute watersheds on coarse
    coarse_basins = DrainageBasins(coarse)
    print(f"  Coarse watersheds: {len(coarse_basins.sheds)}")
    for i, ws in enumerate(coarse_basins.sheds[:5]):
        print(f"    WS {i}: {len(ws.region.hexes)} hexes, ocean={ws.is_ocean}")
    
    # Save coarse watersheds
    save_ws = self.server.save_watersheds(coarse, coarse_basins.sheds, generator.coarse_world_id)
    print(f"  Saved: {save_ws.context}")
    
    # Generate a detail chunk with weather
    origin_chunk = generator.world_to_chunk_ref(HexPosition(0, 0, 0))
    detail = generator.generate_chunk(origin_chunk, coarse)
    generator.compute_chunk_weather(detail)
    
    print(f"\nDetail chunk: {len(detail.hexGrid.hexes)} hexes")
    print(f"  Precip range: {detail.fields['precipitation'].min():.0f} - {detail.fields['precipitation'].max():.0f} mm")
    
    # Compute watersheds on detail
    detail_basins = DrainageBasins(detail)
    print(f"  Detail watersheds: {len(detail_basins.sheds)}")
    for i, ws in enumerate(detail_basins.sheds[:5]):
        print(f"    WS {i}: {len(ws.region.hexes)} hexes, ocean={ws.is_ocean}")
    
    # Save detail chunk and its watersheds
    chunk_id = generator.save_chunk(detail, origin_chunk)
    save_detail_ws = self.server.save_watersheds(detail, detail_basins.sheds, chunk_id)
    print(f"  Saved: {save_detail_ws.context}")
    
    # Visualize both
    print("\n--- Coarse Map ---")
    coarse.hexGrid.adjustRadius(8)
    coarse.colorMap()
    coarse.hexGrid.update()
    coarse.builder.adjust("rivers", coarse_basins.draw_watersheds(top_n=5))
    #display(coarse.builder.show())
    
    print("\n--- Detail Chunk ---")
    detail.hexGrid.adjustRadius(8)
    detail.colorLevels = coarse.colorLevels
    detail.seaLevel = coarse.seaLevel
    detail.colorMap()
    detail.hexGrid.update()
    detail.builder.adjust("rivers", detail_basins.draw_watersheds(top_n=5))
    #display(detail.builder.show())
    
    # Verify we can load them back
    load_coarse_ws = self.server.load_watersheds(generator.coarse_world_id, coarse)
    load_detail_ws = self.server.load_watersheds(chunk_id, detail)
    
    print(f"\n✓ Loaded coarse watersheds: {len(load_coarse_ws.data)}")
    print(f"✓ Loaded detail watersheds: {len(load_detail_ws.data)}")


In [ ]:
@patch
def test_plate_id_roundtrip(self: GeoStorageDebugger):
    """Verify plate_id persists through save/load."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Ensure plate_id exists
    if 'plate_id' not in terrain.fields:
        terrain.fields['plate_id'] = np.random.randint(0, 5, len(terrain.elevations))
    
    original_plates = terrain.fields['plate_id'].copy()
    
    # Save and load
    result = self.server.save_world(terrain, name="Plate Test")
    assert result.status == 'saved'
    
    loaded = self.server.load_world(result.id)
    assert loaded.status == 'loaded'
    
    # Verify plate_id restored
    assert 'plate_id' in loaded.data.fields, "plate_id not in loaded fields"
    assert np.array_equal(loaded.data.fields['plate_id'], original_plates), "plate_id mismatch"
    print(f"✓ plate_id round-trip: {len(np.unique(original_plates))} unique plates preserved")


can you build tests for load_chunk and save_chunk

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a test for `save_chunk` and `load_chunk`:

```python
@patch
def test_save_load_chunk(self: GeoStorageDebugger):
    """Test saving and loading HexChunk."""
    rings = 5
    position = HexPosition(2, -1, -1)
    
    # Create a chunk
    chunk = HexChunk(position, rings)
    chunk.elevations = np.random.rand(len(chunk.elevations)) * 100
    chunk.add_field('plate_id', default=-1)
    chunk.fields['plate_id'] = np.random.randint(0, 4, len(chunk.elevations))
    
    original_elevations = chunk.elevations.copy()
    original_plates = chunk.fields['plate_id'].copy()
    
    # Save
    chunk_ref = ChunkRef(position, rings)
    result = self.server.save_chunk(chunk)
    assert result.status == 'saved', f"Save failed: {result.context}"
    print(f"✓ Saved chunk: {result.context}")
    
    # Load
    loaded_result = self.server.load_chunk(chunk_ref)
    assert loaded_result.status == 'loaded', f"Load failed: {loaded_result.context}"
    loaded = loaded_result.data
    print(f"✓ Loaded chunk: {loaded_result.context}")
    
    # Verify position
    assert loaded.position == position, f"Position mismatch: {loaded.position} vs {position}"
    print(f"✓ Position matches: {loaded.position}")
    
    # Verify elevations (core only, since save_chunk only saves core)
    for idx in chunk.iter_core():
        assert np.isclose(loaded.elevations[idx], original_elevations[idx]), \
            f"Elevation mismatch at {idx}"
    print(f"✓ Core elevations match")
    
    # Verify plate_id
    if 'plate_id' in loaded.fields:
        for idx in chunk.iter_core():
            assert loaded.fields['plate_id'][idx] == original_plates[idx], \
                f"plate_id mismatch at {idx}"
        print(f"✓ plate_id matches")
    else:
        print("⚠ plate_id not restored (check load_chunk)")
```

A few things I noticed that might need attention:

1. **`save_chunk` takes a `HexChunk`** but doesn't take the `ChunkRef` directly - it builds the name from `chunk.position`
2. **`load_chunk` takes a `ChunkRef`** and builds the name from `chunk_ref.position`

Want me to also add a test for multiple chunks or chunk-to-world coordinate mapping?

build a test for chunk to world coordinate mapping using HexChunks

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def test_chunk_world_coordinate_mapping(self: GeoStorageDebugger):
    """Test chunk-to-world and world-to-chunk coordinate mapping."""
    chunk_rings = 5
    spacing = chunk_rings * 2  # Expected spacing between chunk centers
    
    # Test 1: Origin chunk
    origin_ref = ChunkRef(HexPosition(0, 0, 0), chunk_rings)
    assert origin_ref.center_hex == HexPosition(0, 0, 0), "Origin chunk center should be (0,0,0)"
    print(f"✓ Origin chunk center: {origin_ref.center_hex}")
    
    # Test 2: Adjacent chunk
    adj_ref = ChunkRef(HexPosition(1, 0, -1), chunk_rings)
    expected_center = HexPosition(spacing, 0, -spacing)
    assert adj_ref.center_hex == expected_center, f"Expected {expected_center}, got {adj_ref.center_hex}"
    print(f"✓ Adjacent chunk (1,0,-1) center: {adj_ref.center_hex}")
    
    # Test 3: Local to world mapping
    local_pos = HexPosition(2, -1, -1)
    world_pos = chunk_to_world(origin_ref, local_pos)
    assert world_pos == local_pos, "Origin chunk local should equal world"
    print(f"✓ chunk_to_world at origin: {local_pos} -> {world_pos}")
    
    # Test 4: World to chunk round-trip
    test_world = HexPosition(15, -8, -7)
    chunk_pos, local_pos = world_to_chunk(test_world, chunk_rings)
    reconstructed_ref = ChunkRef(chunk_pos, chunk_rings)
    reconstructed_world = chunk_to_world(reconstructed_ref, local_pos)
    assert reconstructed_world == test_world, f"Round-trip failed: {test_world} -> {reconstructed_world}"
    print(f"✓ Round-trip: {test_world} -> chunk {chunk_pos}, local {local_pos} -> {reconstructed_world}")
    
    # Test 5: Multiple positions round-trip
    test_positions = [
        HexPosition(0, 0, 0),
        HexPosition(5, -2, -3),
        HexPosition(-12, 7, 5),
        HexPosition(25, -10, -15),
    ]
    for wp in test_positions:
        cp, lp = world_to_chunk(wp, chunk_rings)
        ref = ChunkRef(cp, chunk_rings)
        result = chunk_to_world(ref, lp)
        assert result == wp, f"Failed for {wp}"
    print(f"✓ All {len(test_positions)} round-trip tests passed")
```

This tests:
1. Origin chunk center is at (0,0,0)
2. Adjacent chunk centers are spaced correctly
3. `chunk_to_world` mapping
4. `world_to_chunk` inverse mapping
5. Round-trip for multiple positions

Want me to add tests that also verify the mapping works with actual `HexChunk` objects and their `hexposition_to_index` method?

So hex chunks have a spiral mapping system for their arrays with the outer part being the halo. can you test that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good idea! Before I write the test, let me make sure I understand the structure correctly:

1. **Core hexes** - the inner part, accessed via `iter_core()`
2. **Halo hexes** - the outer ring(s), presumably indices beyond the core
3. **Spiral ordering** - indices increase in a spiral pattern from center outward?

A few questions:
- Does `HexChunk` have a method like `iter_halo()` to iterate halo indices, or do you just use indices >= `core_size`?
- Is the spiral center at index 0, or somewhere else?
- What's the relationship between `rings` and `core_size`?

Let me know and I'll build a test that verifies the mapping is consistent!

In [ ]:
??HexChunk

If you look at the initializer:
```
# Build/retrieve spiral mapping
        if rings not in HexChunk._spiral_cache:
            spiral = HexPosition.origin().spiral(rings)
            pos_to_idx = {pos: i for i, pos in enumerate(spiral)}
            HexChunk._spiral_cache[rings] = (spiral, pos_to_idx)

        self._spiral, self._pos_to_idx = HexChunk._spiral_cache[rings]

        # Core/halo boundary - indices < core_size are core
        self.core_size = core_size if core_size is not None else len(self._spiral)

        # Data arrays indexed by spiral order
        self.elevations = np.zeros(len(self._spiral))
        self.fields: dict[str, np.ndarray] = {}
``` 
you can see how it goes outward through a spiral the core size is the cutoff element of a spiral and should be the first element of the ring. This neatly divides the chunk into a local and halo part. 

But the math is a bit unorthodox especailly compared to grid traversing. It does have the nice procerty of you being adjacent to the index above and below you, but you have no guarantee of their directions.

Is this something you factored into load_chunk, save_chunk and the tests?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right to flag this. Looking at the code:

**`save_chunk`** stores `local_pos.q/r/s` (from `index_to_hexposition`) plus `grid_index: idx`

**`load_chunk`** uses `hexposition_to_index(pos)` to reconstruct the spiral index from the stored position

This should work because the mapping is position-based, not index-based. But the test should verify this explicitly:

```python
@patch
def test_chunk_spiral_mapping(self: GeoStorageDebugger):
    """Test HexChunk spiral indexing and core/halo boundaries."""
    core_rings = 3
    halo_rings = 2
    position = HexPosition(0, 0, 0)
    
    chunk = HexChunk.with_halo(position, core_rings, halo_rings)
    
    # Test 1: Verify spiral size formula
    expected_core_size = HexChunk.spiral_size(core_rings)
    expected_total = HexChunk.spiral_size(core_rings + halo_rings)
    assert chunk.core_size == expected_core_size, f"Core size: {chunk.core_size} vs {expected_core_size}"
    assert len(chunk) == expected_total, f"Total size: {len(chunk)} vs {expected_total}"
    print(f"✓ Spiral sizes: core={chunk.core_size}, total={len(chunk)}")
    
    # Test 2: Index 0 is always origin
    center_pos = chunk.index_to_hexposition(0)
    assert center_pos == HexPosition(0, 0, 0), f"Index 0 should be origin, got {center_pos}"
    print(f"✓ Index 0 is origin")
    
    # Test 3: Bidirectional mapping is consistent
    for idx in range(len(chunk)):
        pos = chunk.index_to_hexposition(idx)
        back_idx = chunk.hexposition_to_index(pos)
        assert back_idx == idx, f"Round-trip failed: {idx} -> {pos} -> {back_idx}"
    print(f"✓ All {len(chunk)} indices round-trip through position")
    
    # Test 4: Core/halo boundary
    last_core_idx = chunk.core_size - 1
    first_halo_idx = chunk.core_size
    
    assert chunk.is_core(last_core_idx), f"Index {last_core_idx} should be core"
    assert chunk.is_halo(first_halo_idx), f"Index {first_halo_idx} should be halo"
    
    last_core_pos = chunk.index_to_hexposition(last_core_idx)
    first_halo_pos = chunk.index_to_hexposition(first_halo_idx)
    
    # Core boundary should be at distance == core_rings from center
    assert last_core_pos.distance(HexPosition(0,0,0)) == core_rings, \
        f"Last core hex should be at distance {core_rings}"
    assert first_halo_pos.distance(HexPosition(0,0,0)) == core_rings + 1, \
        f"First halo hex should be at distance {core_rings + 1}"
    print(f"✓ Core/halo boundary at ring {core_rings}")
    
    # Test 5: Save/load preserves position-based data
    # Set unique values at specific positions
    test_positions = [
        HexPosition(0, 0, 0),      # center
        HexPosition(1, -1, 0),     # ring 1
        HexPosition(core_rings, -core_rings, 0),  # core edge
    ]
    
    for i, pos in enumerate(test_positions):
        idx = chunk.hexposition_to_index(pos)
        chunk.elevations[idx] = (i + 1) * 100.0
    
    # Save
    result = self.server.save_chunk(chunk)
    assert result.status == 'saved', f"Save failed: {result.context}"
    
    # Load into fresh chunk
    chunk_ref = ChunkRef(position, core_rings + halo_rings)
    loaded_result = self.server.load_chunk(chunk_ref)
    assert loaded_result.status == 'loaded'
    loaded = loaded_result.data
    
    # Verify by position, not index
    for i, pos in enumerate(test_positions):
        loaded_idx = loaded.hexposition_to_index(pos)
        expected = (i + 1) * 100.0
        actual = loaded.elevations[loaded_idx]
        assert np.isclose(actual, expected), \
            f"Position {pos}: expected {expected}, got {actual}"
    print(f"✓ Position-based data preserved through save/load")
```

This tests that:
1. Spiral size formulas are correct
2. Index 0 is always the center
3. idx → pos → idx round-trips
4. Core/halo boundary aligns with ring distance
5. Data at *positions* (not indices) survives save/load

In [ ]:
#| export
@patch
def test_chunk_spiral_mapping(self: GeoStorageDebugger):
    """Test HexChunk spiral indexing and core/halo boundaries."""
    core_rings = 3
    halo_rings = 2
    position = HexPosition(0, 0, 0)
    
    chunk = HexChunk.with_halo(position, core_rings, halo_rings)
    
    # Test 1: Verify spiral size formula
    expected_core_size = HexChunk.spiral_size(core_rings)
    expected_total = HexChunk.spiral_size(core_rings + halo_rings)
    assert chunk.core_size == expected_core_size, f"Core size: {chunk.core_size} vs {expected_core_size}"
    assert len(chunk) == expected_total, f"Total size: {len(chunk)} vs {expected_total}"
    print(f"✓ Spiral sizes: core={chunk.core_size}, total={len(chunk)}")
    
    # Test 2: Index 0 is always origin
    center_pos = chunk.index_to_hexposition(0)
    assert center_pos == HexPosition(0, 0, 0), f"Index 0 should be origin, got {center_pos}"
    print(f"✓ Index 0 is origin")
    
    # Test 3: Bidirectional mapping is consistent
    for idx in range(len(chunk)):
        pos = chunk.index_to_hexposition(idx)
        back_idx = chunk.hexposition_to_index(pos)
        assert back_idx == idx, f"Round-trip failed: {idx} -> {pos} -> {back_idx}"
    print(f"✓ All {len(chunk)} indices round-trip through position")
    
    # Test 4: Core/halo boundary
    last_core_idx = chunk.core_size - 1
    first_halo_idx = chunk.core_size
    
    assert chunk.is_core(last_core_idx), f"Index {last_core_idx} should be core"
    assert chunk.is_halo(first_halo_idx), f"Index {first_halo_idx} should be halo"
    
    last_core_pos = chunk.index_to_hexposition(last_core_idx)
    first_halo_pos = chunk.index_to_hexposition(first_halo_idx)
    
    # Core boundary should be at distance == core_rings from center
    assert last_core_pos.distance(HexPosition(0,0,0)) == core_rings, \
        f"Last core hex should be at distance {core_rings}"
    assert first_halo_pos.distance(HexPosition(0,0,0)) == core_rings + 1, \
        f"First halo hex should be at distance {core_rings + 1}"
    print(f"✓ Core/halo boundary at ring {core_rings}")
    
    # Test 5: Save/load preserves position-based data
    # Set unique values at specific positions
    test_positions = [
        HexPosition(0, 0, 0),      # center
        HexPosition(1, -1, 0),     # ring 1
        HexPosition(core_rings, -core_rings, 0),  # core edge
    ]
    
    for i, pos in enumerate(test_positions):
        idx = chunk.hexposition_to_index(pos)
        chunk.elevations[idx] = (i + 1) * 100.0
    
    # Save
    result = self.server.save_chunk(chunk)
    assert result.status == 'saved', f"Save failed: {result.context}"
    
    # Load into fresh chunk
    chunk_ref = ChunkRef(position, core_rings + halo_rings)
    loaded_result = self.server.load_chunk(chunk_ref)
    assert loaded_result.status == 'loaded'
    loaded = loaded_result.data
    
    # Verify by position, not index
    for i, pos in enumerate(test_positions):
        loaded_idx = loaded.hexposition_to_index(pos)
        expected = (i + 1) * 100.0
        actual = loaded.elevations[loaded_idx]
        assert np.isclose(actual, expected), \
            f"Position {pos}: expected {expected}, got {actual}"
    print(f"✓ Position-based data preserved through save/load")


In [ ]:
#| export
@patch
def test_chunk_halo_overlap(self: GeoStorageDebugger):
    """Test that adjacent chunks' halos overlap correctly."""
    core_rings = 3
    halo_rings = 2
    
    # Two adjacent chunks
    chunk_a = HexChunk.with_halo(HexPosition(0, 0, 0), core_rings, halo_rings)
    chunk_b = HexChunk.with_halo(HexPosition(1, -1, 0), core_rings, halo_rings)
    
    # Set elevations in chunk_a
    for idx in chunk_a.iter_core():
        chunk_a.elevations[idx] = idx * 10.0
    
    # Copy overlapping data to chunk_b's halo
    chunk_b.copy_from(chunk_a)
    
    # Find hexes that should overlap (chunk_b's halo ∩ chunk_a's core)
    overlap_count = 0
    for idx in chunk_b.iter_halo():
        world_pos = chunk_b.index_to_world(idx)
        a_idx = chunk_a.world_to_index(world_pos)
        if a_idx >= 0 and chunk_a.is_core(a_idx):
            assert chunk_b.elevations[idx] == chunk_a.elevations[a_idx]
            overlap_count += 1
    
    print(f"✓ {overlap_count} halo hexes copied from adjacent chunk")


In [ ]:
#| export
@patch
def test_load_chunk_gets_latest(self: GeoStorageDebugger):
    """Test that load_chunk retrieves the most recent version when multiple exist."""
    
    position = HexPosition(0, 0, 0)
    core_rings = 3
    halo_rings = 2
    
    # Create and save first chunk
    chunk1 = HexChunk.with_halo(position, core_rings, halo_rings)
    chunk1.elevations[0] = 100.0  # Center elevation
    chunk1.elevations[1] = 111.0  # Marker value
    
    result1 = self.server.save_chunk(chunk1)
    assert result1.status == 'saved'
    print(f"✓ Saved chunk v1 with center elevation 100.0")
    
    # Small delay to ensure different timestamp (or we could mock time)
    import time
    time.sleep(0.1)
    
    # Create and save second chunk at SAME position with different data
    chunk2 = HexChunk.with_halo(position, core_rings, halo_rings)
    chunk2.elevations[0] = 200.0  # Different center elevation
    chunk2.elevations[1] = 222.0  # Different marker
    
    result2 = self.server.save_chunk(chunk2)
    assert result2.status == 'saved'
    assert result2.id != result1.id, "Should create new world entry"
    print(f"✓ Saved chunk v2 with center elevation 200.0 (different world_id)")
    
    # Load chunk - should get the LATEST (v2)
    chunk_ref = ChunkRef(position, core_rings + halo_rings)
    load_result = self.server.load_chunk(chunk_ref)
    
    assert load_result.status == 'loaded'
    loaded = load_result.data
    
    # Verify we got v2, not v1
    assert np.isclose(loaded.elevations[0], 200.0), \
        f"Expected 200.0 (v2), got {loaded.elevations[0]} - loaded old version!"
    assert np.isclose(loaded.elevations[1], 222.0), \
        f"Expected 222.0 (v2), got {loaded.elevations[1]}"
    print(f"✓ load_chunk returned latest version (elevation={loaded.elevations[0]})")


## Run it

In [ ]:
with GeoStorageDebugger(keep_on_error=False) as dbg:
    #dbg.test_multiple_worlds()
    #dbg.test_multiscale_watersheds()
    #dbg.test_smallWs()
    dbg.test_save_load_terrain()
    dbg.test_query_radius()
    dbg.test_extract_region()
    dbg.test_region_mappers()
    dbg.test_save_load_weather()
    #dbg.test_chunked_terrain_generator()
    dbg.test_save_load_watersheds()
    dbg.test_plate_id_roundtrip()
    dbg.test_chunk_spiral_mapping()
    dbg.test_chunk_halo_overlap()
    dbg.test_load_chunk_gets_latest()


any insight? didn't seem like the test worked

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The bug is in `load_chunk`'s query. Looking at the SQL:

```python
"SELECT * FROM terrain_world WHERE name = ? ORDER BY created DESC LIMIT 1"
```

The issue: `created` is `int(datetime.now().timestamp())` which has **second precision**. Your 0.1 second sleep means both chunks likely have the same `created` value, so the order is non-deterministic.

**Fix**: Use `id DESC` instead since auto-increment IDs are guaranteed to increase:

```python
@patch  
def load_chunk(self: GeoStorage, chunk_ref: ChunkRef) -> LoadResult:
    """Load HexChunk from database."""
    p = chunk_ref.position
    name = f"chunk_{p.q}_{p.r}_{p.s}"
    
    # Find the most recent chunk by name - use id DESC since created has second precision
    cursor = self.db.execute(
        "SELECT * FROM terrain_world WHERE name = ? ORDER BY id DESC LIMIT 1",
        [name]
    )
    # ... rest unchanged
```

Alternatively, if you want to keep using `created`, you could change timestamps to microseconds:

```python
now = int(datetime.now().timestamp() * 1_000_000)  # microseconds
```

But `id DESC` is simpler and doesn't require schema changes.

## Real Data

In [ ]:
with open('sf_coast_weather.txt') as f:
    sfTerr = Terrain.decode(f.read())
sfTerr.hexGrid.adjustRadius(10)
sfTerr.colorMap()
sfTerr.hexGrid.update()
len(sfTerr.builder.xml())


How much bigger can these chunked worlds be given our 3-6k size limit on view ports


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question! The chunked architecture decouples **world size** from **viewport size**.

**Viewport constraint**: ~3-6k rendered hexes at once (based on your XML/SVG limits)

**Chunk sizing math**:
- A centered hex grid with `rings=15` has ~721 hexes (formula: `3*r² + 3*r + 1`)
- At `fine_radius=10`, this fits comfortably in viewport
- Each chunk renders independently

**World scaling**:
| Chunk rings | Chunks in world | Total hexes | Notes |
|-------------|-----------------|-------------|-------|
| 7 rings of chunks | ~127 chunks | ~90k hexes | Small continent |
| 15 rings of chunks | ~721 chunks | ~500k hexes | Large region |
| 50 rings of chunks | ~7,651 chunks | ~5.5M hexes | Full planet slice |

**Practical limits**:
- **Storage**: SQLite handles millions of rows easily
- **Generation**: On-demand, so only visited chunks pay the cost
- **Cross-chunk operations** (pathfinding, rivers): This is the real constraint - need to load multiple chunks into memory

**Recommendation**: 
- Coarse map: ~100-500 hexes (fits in single viewport for overview)
- Detail chunks: ~500-700 hexes each
- Total world: **100k-1M hexes** is very practical
- Beyond that, consider hierarchical chunking (chunks of chunks)

The coarse-to-fine ratio of 5:1 you're using is good - gives 25x more detail hexes than coarse overview hexes.

So maybe we store halo_rings?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes—storing `halo_rings` per chunk tells you how much overlap you have for edge computations.

You might also consider storing precomputed **edge connectivity** directly:
- Which border hexes connect across chunk boundaries
- Traversal costs between them

That way pathfinding can query the chunk table for connectivity without loading the full hex data. Would you want that as a separate table (like `ChunkEdge`) or a JSON blob in the `Chunk` row?

Hmm back to my pondering. Is there anything like https://observablehq.com/@nrabinowitz/h3-pathfinding or https://www.redblobgames.com/pathfinding/a-star/introduction.html that would work. I sort of like upsampling to get a general route and then downsample as needed to get specific hexes on smaller maps that you connect. It might not be fully optimal, but probably pretty good. Is there anything we could do for the database that would help.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Your intuition is solid—hierarchical pathfinding (coarse first, refine locally) is a well-established approach. Red Blob Games calls it "HPA*" (Hierarchical Pathfinding A*).

For database support, a few things could help:

1. **Chunk adjacency table** — precompute which chunks neighbor which, so coarse pathfinding can query neighbors without geometry math

2. **Border hex table** — store the hexes at chunk edges with their world coordinates, so you can quickly find where to "hand off" between chunks

3. **Precomputed costs** — for each chunk, store min/max traversal cost (or average). This lets the coarse A* heuristic be more accurate.

Which of these feels most useful for your use case? Or would you rather sketch out the pathfinding algorithm first and let the schema follow from that?

So we have made great progress on a great deal of the steps TerrainSchema.md. I want to put in a section about what we have built so far. are you able to cat to the end of the file. if not put it below and I can add it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I can't write to files directly. Here's the section to append to the end of `schemaMain.md`:

```markdown
---

## Implementation Progress

### Completed (2026-02-01)

#### Core Classes

**GeoStorage** - Main database manager
- `__init__(custom_path)` - Initialize with FastLite database
- `get_db_path()` - Package data or user fallback
- `reset_database()` - Delete and recreate (testing)
- `_get(obj, key)` - Dict/object attribute accessor

**Tables Created**
- `TerrainWorld` - World metadata (name, hex_radius, nrows, ncols, sea_level, elevation_delta, extras JSON)
- `HexData` - Per-hex spatial data (world_id, q, r, s, grid_index, elevation, distance_from_coast, modified timestamp)
- `User` - User accounts with activeWorld reference

**Indices**
- `idx_hex_world` - world_id lookup
- `idx_hex_coords` - (world_id, q, r, s) spatial queries
- `idx_hex_grid` - (world_id, grid_index) array-order queries

#### Save/Load Operations

**save_world(terrain, name)** → SaveResult
- Extracts grid parameters (radius, nrows, ncols)
- Stores geo/climate in extras JSON
- Batch inserts all HexData rows with elevation + distance_from_coast
- Returns SaveResult(world_id, status, context)

**load_world(world_id)** → LoadResult
- Reconstructs Terrain from TerrainWorld metadata
- Rebuilds grid with `adjustRadius()`
- Populates elevations and fields from HexData
- Returns LoadResult(terrain, status, context)

#### Spatial Queries

**query_hexes_in_radius(world_id, center_q, center_r, center_s, radius)** → list
- Bounding box SQL query followed by hex distance filter
- Returns list of HexData rows

**extract_region(world_id, center_q, center_r, center_s, rings, padding)** → LoadResult
- Creates centered HexGrid via `HexGrid.centered()`
- Maps DB rows to new grid using HexPosition arithmetic
- Returns LoadResult(terrain, status, context) with centered Terrain

#### Index Mappers

**create_region_coord_mapper(center_q, center_r, center_s, extracted_grid)** → callable
- Returns function: `new_idx → (q, r, s)` in world coordinates

**create_region_index_mapper(world_id, center_q, center_r, center_s, extracted_grid)** → callable
- Builds (q,r,s) → grid_index lookup from DB
- Returns function: `new_idx → original_grid_index` (or -1)

#### Testing

**GeoStorageDebugger** - Test harness with temp database
- Auto-cleanup unless `keep_on_error=True`
- `preserve_db(test_name)` - Copy to ~/.hexmagic/test_failures/

**Tests Passing**
- `test_save_load_terrain()` - Round-trip with elevation verification
- `test_query_radius()` - Spatial query returns expected hexes
- `test_extract_region()` - Centered grid extraction
- `test_region_mappers()` - Coordinate and index mapping

### Next Steps

1. **Downsampling** - `save_downsampled_world()` with ScaleTransform tracking
2. **Print rectangles** - `extract_rectangle()` for landscape/portrait crops
3. **Viewport persistence** - Save/load named viewports
4. **Additional fields** - Temperature, precipitation, watersheds
5. **Template system** - Pre-populate with TerraDemo maps
```

#| export
and how would you update llms.txt

